# Practical 6 — Domain Shift: MegaDetector on Aerial Imagery

**Context:** MegaDetector was trained on millions of **camera trap** images —
side-view photos where animals are typically large, close, and at eye level.
What happens when we apply it to **aerial drone imagery** where animals are
tiny (30–60 px), viewed from above (nadir), and surrounded by savanna landscape?

This notebook uses the **Eikelboom et al. (2019)** dataset — aerial drone surveys
of elephants, zebras, and giraffes in Kenya — to quantify the domain shift.
We run MegaDetector v1000 (larch) on these images and compare its predictions
against the ground truth bounding boxes.

| Property | Camera trap (MD training data) | Eikelboom aerial |
|----------|-------------------------------|------------------|
| Viewpoint | Side-view, eye-level | Top-down (nadir) |
| Animal size | 100–1000+ px | 30–60 px |
| Image size | ~2 MP | ~14 MP (4603 x 3068) |
| Background | Forest floor, trails | Savanna, bare soil |
| Species appearance | Side profile | Top-down silhouette |

**Expected outcome:** MegaDetector will miss most animals — demonstrating
why domain-specific models (like the Eikelboom RetinaNet or HerdNet) exist.

**Reference:**
> Eikelboom et al. (2019). Improving the precision and accuracy of animal population
> estimates with aerial image object detection. *Methods in Ecology and Evolution*, 10(11).
> [DOI: 10.1111/2041-210X.13277](https://doi.org/10.1111/2041-210X.13277)

---

## Environment Setup

**Local (recommended):** use the `fit-training` conda environment:

```bash
conda env create -f environment-training.yml
conda activate fit-training
```

**Google Colab:** uncomment and run the cell below.

In [ ]:
# Colab only — install dependencies if not already available
# import sys

# !git clone -b course_draft https://github.com/cwinkelmann/usde-innovations-applications-forest-it.git fit-course
# !cd fit-course && git pull
# !pip install -e "./fit-course[training,dev]"
#
# sys.path.append('./fit-course')

In [ ]:
%matplotlib inline

from pathlib import Path
import json
import time

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

# ── Paths ──────────────────────────────────────────────────────────────────
DATA_DIR = Path("../data")
EIKELBOOM_DIR = DATA_DIR / "eikelboom"
TEST_IMG_DIR = EIKELBOOM_DIR / "test"
ANNOTATION_CSV = EIKELBOOM_DIR / "annotations" / "annotations_test.csv"
OUTPUT_DIR = DATA_DIR / "domain_shift_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TEST_IMG_DIR.exists(), f"Eikelboom test images not found at {TEST_IMG_DIR}"
assert ANNOTATION_CSV.exists(), f"Annotations not found at {ANNOTATION_CSV}"

print(f"Test images  : {TEST_IMG_DIR}")
print(f"Annotations  : {ANNOTATION_CSV}")
print(f"Output       : {OUTPUT_DIR}")

## 1 — Load ground truth

The Eikelboom annotations are CSV files with **pixel-level bounding boxes**
in Pascal VOC format: `filename, x1, y1, x2, y2, species`.

Three species: Elephant, Zebra, Giraffe — all viewed from above.

In [ ]:
gt = pd.read_csv(ANNOTATION_CSV, header=None,
                 names=["file", "x1", "y1", "x2", "y2", "species"])

# Strip path prefix — annotations have 'test/filename.JPG'
gt["filename"] = gt["file"].apply(lambda x: Path(x).name)
gt["w"] = gt["x2"] - gt["x1"]
gt["h"] = gt["y2"] - gt["y1"]

print(f"Ground truth: {len(gt)} annotations across {gt['filename'].nunique()} images")
print(f"\nSpecies counts:")
print(gt["species"].value_counts())
print(f"\nBox sizes (pixels):")
print(gt[["w", "h"]].describe().round(1))

In [ ]:
# Visualize a few images with ground truth
sample_files = gt["filename"].unique()[:4]

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
for ax, fname in zip(axes.flat, sample_files):
    img = Image.open(TEST_IMG_DIR / fname)
    ax.imshow(np.array(img))

    img_gt = gt[gt["filename"] == fname]
    colors = {"Elephant": "#E74C3C", "Zebra": "#3498DB", "Giraffe": "#F39C12"}
    for _, row in img_gt.iterrows():
        c = colors.get(row["species"], "white")
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1, edgecolor=c, facecolor="none",
        ))

    ax.set_title(f"{fname} — {len(img_gt)} animals ({img.size[0]}x{img.size[1]} px)", fontsize=9)
    ax.axis("off")

legend_patches = [mpatches.Patch(color=c, label=s) for s, c in colors.items()]
fig.legend(handles=legend_patches, loc="lower center", ncol=3, fontsize=10)
plt.suptitle("Eikelboom aerial dataset — ground truth", fontsize=13)
plt.tight_layout()

In [ ]:
# Zoom into individual animals to see what MD has to work with
# Pick one image with many annotations and crop around each GT box
dense_file = gt.groupby("filename").size().sort_values(ascending=False).index[0]
dense_gt = gt[gt["filename"] == dense_file]
full_img = np.array(Image.open(TEST_IMG_DIR / dense_file))

sample_rows = dense_gt.sample(min(8, len(dense_gt)), random_state=42)
n_show = len(sample_rows)
fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
if n_show == 1:
    axes = [axes]

pad = 60  # pixels of context around each box
for ax, (_, row) in zip(axes, sample_rows.iterrows()):
    cx = (row["x1"] + row["x2"]) / 2
    cy = (row["y1"] + row["y2"]) / 2
    crop_x1 = max(0, int(cx - pad))
    crop_y1 = max(0, int(cy - pad))
    crop_x2 = min(full_img.shape[1], int(cx + pad))
    crop_y2 = min(full_img.shape[0], int(cy + pad))

    crop = full_img[crop_y1:crop_y2, crop_x1:crop_x2]
    ax.imshow(crop)

    # Draw GT box relative to crop origin
    ax.add_patch(mpatches.Rectangle(
        (row["x1"] - crop_x1, row["y1"] - crop_y1), row["w"], row["h"],
        linewidth=2, edgecolor="#2ECC71", facecolor="none",
    ))
    ax.set_title(f"{row['species']}\n{row['w']}x{row['h']} px", fontsize=8)
    ax.axis("off")

plt.suptitle(f"Zoomed ground truth animals from {dense_file}", fontsize=11)
plt.tight_layout()

Notice how small the animals are relative to the full image.
The average bounding box is ~50 x 58 pixels in a 4603 x 3068 image.
That's **~0.02%** of the image area per animal.

## 2 — Load MegaDetector v1000 (larch)

Same model as in the ultralytics notebook. MD v1000 larch is a YOLOv11-L
trained on camera trap images.

In [ ]:
import os
import torch
import wget
from ultralytics import YOLO

cache_dir = os.path.join(torch.hub.get_dir(), "checkpoints")
weights_path = os.path.join(cache_dir, "md_v1000.0.0-larch.pt")
if not os.path.exists(weights_path):
    os.makedirs(cache_dir, exist_ok=True)
    print("Downloading MD v1000 larch weights (~49 MB)...")
    wget.download(
        "https://github.com/agentmorris/MegaDetector/releases/download/v1000.0/md_v1000.0.0-larch.pt",
        out=weights_path,
    )
    print()

md_model = YOLO(weights_path)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# MD v1000 categories: 0=animal, 1=person, 2=vehicle
MD_LABELS = {0: "animal", 1: "person", 2: "vehicle"}

print(f"MegaDetector v1000 larch loaded")
print(f"Device: {DEVICE}")

## 3 — Run MegaDetector on Eikelboom test images

We run at the **default input resolution** first — the same way you would
run MegaDetector on camera trap images. No SAHI, no tiling, just the
standard `model.predict()` call.

The images are 4603 x 3068 but YOLO internally resizes to 640 px —
each animal shrinks to roughly **4–8 pixels**. Can the model still detect them?

Set `DEMO_MODE = True` to run on a small subset (5 images) for quick iteration.
Set it to `False` to run on the full test set (112 images) for proper evaluation.

In [ ]:
CONF_THRESHOLD = 0.1  # Low threshold to catch anything MD finds
DEMO_MODE = True      # True = 5 images (fast), False = full test set (112 images)

all_test_images = sorted(TEST_IMG_DIR.glob("*.JPG"))

if DEMO_MODE:
    # Pick 5 images with the most annotations for a representative demo
    gt_per_image = gt.groupby("filename").size().sort_values(ascending=False)
    demo_files = set(gt_per_image.head(5).index)
    test_images = [p for p in all_test_images if p.name in demo_files]
    print(f"DEMO MODE: running on {len(test_images)} of {len(all_test_images)} images")
    print(f"  (selected images with most annotations: {sorted(demo_files)})")
else:
    test_images = all_test_images
    print(f"Running MegaDetector on all {len(test_images)} aerial images...")

# Filter ground truth to match selected images
gt_filenames = {p.name for p in test_images}
gt_subset = gt[gt["filename"].isin(gt_filenames)]
print(f"  Ground truth for selected images: {len(gt_subset)} annotations")

md_results = []
t0 = time.time()

for img_path in test_images:
    results = md_model.predict(str(img_path), conf=CONF_THRESHOLD, verbose=False)
    boxes_obj = results[0].boxes

    dets = []
    for i in range(len(boxes_obj)):
        x1, y1, x2, y2 = boxes_obj.xyxy[i].tolist()
        cls_id = int(boxes_obj.cls[i])
        conf = float(boxes_obj.conf[i])
        dets.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "conf": conf, "class": cls_id,
            "label": MD_LABELS.get(cls_id, "unknown"),
        })

    md_results.append({"filename": img_path.name, "detections": dets})

elapsed = time.time() - t0

total_dets = sum(len(r["detections"]) for r in md_results)
animal_dets = sum(
    sum(1 for d in r["detections"] if d["class"] == 0)
    for r in md_results
)
gt_total = len(gt_subset)

print(f"\nDone in {elapsed:.1f}s")
print(f"Total detections (all classes): {total_dets}")
print(f"Animal detections: {animal_dets}")
print(f"Ground truth animals: {gt_total}")
print(f"\n→ MegaDetector found {animal_dets} of {gt_total} animals "
      f"({100 * animal_dets / gt_total:.1f}% raw recall)")

## 4 — Visual comparison: predictions vs. ground truth

Let's overlay both MegaDetector predictions (red dashed) and ground truth (green solid)
on the same images to see the domain shift visually.

In [ ]:
# Pick images with the most ground truth annotations
gt_counts = gt_subset.groupby("filename").size().sort_values(ascending=False)
show_files = gt_counts.head(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, fname in zip(axes.flat, show_files):
    img = np.array(Image.open(TEST_IMG_DIR / fname))
    ax.imshow(img)

    # Ground truth (green)
    img_gt = gt_subset[gt_subset["filename"] == fname]
    for _, row in img_gt.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1.2, edgecolor="#2ECC71", facecolor="none", linestyle="-",
        ))

    # MegaDetector predictions (red dashed)
    md_img = next((r for r in md_results if r["filename"] == fname), None)
    n_pred = 0
    if md_img:
        for det in md_img["detections"]:
            if det["class"] != 0:  # animals only
                continue
            ax.add_patch(mpatches.Rectangle(
                (det["x1"], det["y1"]),
                det["x2"] - det["x1"], det["y2"] - det["y1"],
                linewidth=1.5, edgecolor="#E74C3C", facecolor="none", linestyle="--",
            ))
            n_pred += 1

    ax.set_title(f"{fname} — GT: {len(img_gt)} | MD: {n_pred}", fontsize=9)
    ax.axis("off")

legend = [
    mpatches.Patch(edgecolor="#2ECC71", facecolor="none", label="Ground truth"),
    mpatches.Patch(edgecolor="#E74C3C", facecolor="none", label="MegaDetector", linestyle="--"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("Domain shift: MegaDetector (camera trap model) on aerial imagery", fontsize=13)
plt.tight_layout()

## 4b — Can SAHI (tiled inference) rescue MegaDetector?

SAHI splits large images into overlapping tiles, runs detection on each tile,
then merges results with NMS. This should help because each tile is closer to
camera-trap scale — the animals become larger relative to the input.

We use 640x640 tiles with 20% overlap.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from tqdm.auto import tqdm

sahi_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path=weights_path,
    confidence_threshold=CONF_THRESHOLD,
    device=DEVICE,
)

SLICE_SIZE = 640
OVERLAP_RATIO = 0.2

print(f"Running SAHI tiled inference on {len(test_images)} images...")
print(f"  Tile size: {SLICE_SIZE}x{SLICE_SIZE}, overlap: {OVERLAP_RATIO:.0%}")
print(f"  (~56 tiles per 4603x3068 image)\n")

sahi_results = []
t0 = time.time()

for img_path in tqdm(test_images, desc="SAHI inference"):
    result = get_sliced_prediction(
        str(img_path),
        sahi_model,
        slice_height=SLICE_SIZE,
        slice_width=SLICE_SIZE,
        overlap_height_ratio=OVERLAP_RATIO,
        overlap_width_ratio=OVERLAP_RATIO,
        verbose=0,
    )

    dets = []
    for pred in result.object_prediction_list:
        bbox = pred.bbox
        cls_id = pred.category.id
        conf = pred.score.value
        dets.append({
            "x1": bbox.minx, "y1": bbox.miny,
            "x2": bbox.maxx, "y2": bbox.maxy,
            "conf": conf, "class": cls_id,
            "label": MD_LABELS.get(cls_id, "unknown"),
        })

    sahi_results.append({"filename": img_path.name, "detections": dets})

sahi_elapsed = time.time() - t0

sahi_animal_dets = sum(
    sum(1 for d in r["detections"] if d["class"] == 0)
    for r in sahi_results
)

print(f"\nDone in {sahi_elapsed:.1f}s (vs {elapsed:.1f}s without SAHI)")
print(f"SAHI animal detections: {sahi_animal_dets}")
print(f"Standard MD detections: {animal_dets}")
print(f"Ground truth animals:   {len(gt)}")

In [ ]:
# Visual comparison: Standard MD vs SAHI on the densest images
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
compare_files = gt_counts.head(4).index.tolist()

for ax, fname in zip(axes.flat, compare_files):
    img_arr = np.array(Image.open(TEST_IMG_DIR / fname))
    ax.imshow(img_arr)

    # Ground truth (green)
    img_gt = gt[gt["filename"] == fname]
    for _, row in img_gt.iterrows():
        ax.add_patch(mpatches.Rectangle(
            (row["x1"], row["y1"]), row["w"], row["h"],
            linewidth=1, edgecolor="#2ECC71", facecolor="none",
        ))

    # SAHI predictions (orange dashed)
    sahi_img = next((r for r in sahi_results if r["filename"] == fname), None)
    n_sahi = 0
    if sahi_img:
        for det in sahi_img["detections"]:
            if det["class"] != 0:
                continue
            ax.add_patch(mpatches.Rectangle(
                (det["x1"], det["y1"]),
                det["x2"] - det["x1"], det["y2"] - det["y1"],
                linewidth=1.5, edgecolor="#F39C12", facecolor="none", linestyle="--",
            ))
            n_sahi += 1

    n_gt = len(img_gt)
    ax.set_title(f"{fname} — GT: {n_gt} | SAHI: {n_sahi}", fontsize=9)
    ax.axis("off")

legend = [
    mpatches.Patch(edgecolor="#2ECC71", facecolor="none", label="Ground truth"),
    mpatches.Patch(edgecolor="#F39C12", facecolor="none", label="MD + SAHI", linestyle="--"),
]
fig.legend(handles=legend, loc="lower center", ncol=2, fontsize=11)
plt.suptitle("SAHI tiled inference: does slicing help MegaDetector on aerial images?", fontsize=13)
plt.tight_layout()

In [ ]:
# Zoom into SAHI detections — what is MD actually detecting from above?
# Pick the image with the most SAHI animal detections
sahi_counts = {r["filename"]: sum(1 for d in r["detections"] if d["class"] == 0) for r in sahi_results}
sahi_dense_file = max(sahi_counts, key=sahi_counts.get)
sahi_dense = next(r for r in sahi_results if r["filename"] == sahi_dense_file)
sahi_dense_img = np.array(Image.open(TEST_IMG_DIR / sahi_dense_file))

animal_dets_list = [d for d in sahi_dense["detections"] if d["class"] == 0]
sample_dets = animal_dets_list[:8]  # first 8 detections
n_show = len(sample_dets)

if n_show > 0:
    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
    if n_show == 1:
        axes = [axes]

    pad = 60
    for ax, det in zip(axes, sample_dets):
        cx = (det["x1"] + det["x2"]) / 2
        cy = (det["y1"] + det["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(sahi_dense_img.shape[1], int(cx + pad))
        crop_y2 = min(sahi_dense_img.shape[0], int(cy + pad))

        crop = sahi_dense_img[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        # Draw SAHI detection box relative to crop
        bw = det["x2"] - det["x1"]
        bh = det["y2"] - det["y1"]
        ax.add_patch(mpatches.Rectangle(
            (det["x1"] - crop_x1, det["y1"] - crop_y1), bw, bh,
            linewidth=2, edgecolor="#F39C12", facecolor="none", linestyle="--",
        ))
        ax.set_title(f"{det['label']} {det['conf']:.2f}\n{bw:.0f}x{bh:.0f} px", fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Zoomed SAHI detections from {sahi_dense_file}", fontsize=11)
    plt.tight_layout()
else:
    print(f"No SAHI animal detections to display.")

In [ ]:
# Zoom into ONLY high-confidence SAHI detections (conf >= 0.5)
# These are what the model is "most sure" about — are they real animals or false positives?
HIGH_CONF = 0.5

high_conf_dets = []
for r in sahi_results:
    for d in r["detections"]:
        if d["class"] == 0 and d["conf"] >= HIGH_CONF:
            high_conf_dets.append({**d, "filename": r["filename"]})

print(f"High-confidence SAHI detections (conf >= {HIGH_CONF}): {len(high_conf_dets)}")

n_show = min(8, len(high_conf_dets))
if n_show > 0:
    # Sort by confidence descending
    high_conf_dets.sort(key=lambda d: d["conf"], reverse=True)
    show_dets = high_conf_dets[:n_show]

    fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 3))
    if n_show == 1:
        axes = [axes]

    pad = 60
    for ax, det in zip(axes, show_dets):
        img_arr = np.array(Image.open(TEST_IMG_DIR / det["filename"]))
        cx = (det["x1"] + det["x2"]) / 2
        cy = (det["y1"] + det["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(img_arr.shape[1], int(cx + pad))
        crop_y2 = min(img_arr.shape[0], int(cy + pad))

        crop = img_arr[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        bw = det["x2"] - det["x1"]
        bh = det["y2"] - det["y1"]
        ax.add_patch(mpatches.Rectangle(
            (det["x1"] - crop_x1, det["y1"] - crop_y1), bw, bh,
            linewidth=2, edgecolor="#E74C3C", facecolor="none",
        ))
        ax.set_title(f"conf={det['conf']:.2f}\n{bw:.0f}x{bh:.0f} px", fontsize=8)
        ax.axis("off")

    plt.suptitle(f"Highest-confidence SAHI detections (conf >= {HIGH_CONF}) — real animals or false positives?", fontsize=11)
    plt.tight_layout()
else:
    print(f"No detections above conf {HIGH_CONF}. Try lowering the threshold.")

In [ ]:
# False negatives: animals that NEITHER standard MD nor SAHI detected
# These illustrate the core domain shift problem

def get_detected_gt_indices(gt_df, predictions, iou_threshold=0.3):
    """Return set of GT row indices that were matched by at least one prediction."""
    matched = set()
    for fname in gt_df["filename"].unique():
        img_gt = gt_df[gt_df["filename"] == fname]
        gt_boxes = img_gt[["x1", "y1", "x2", "y2"]].values.tolist()
        gt_indices = img_gt.index.tolist()

        pred_entry = next((r for r in predictions if r["filename"] == fname), None)
        if not pred_entry:
            continue
        pred_boxes = [[d["x1"], d["y1"], d["x2"], d["y2"]]
                      for d in pred_entry["detections"] if d["class"] == 0]

        for pb in pred_boxes:
            best_iou, best_idx = 0, -1
            for j, gb in enumerate(gt_boxes):
                iou = compute_iou(pb, gb)
                if iou > best_iou and gt_indices[j] not in matched:
                    best_iou = iou
                    best_idx = gt_indices[j]
            if best_iou >= iou_threshold and best_idx >= 0:
                matched.add(best_idx)
    return matched

# Find GT annotations missed by both standard MD and SAHI
md_detected = get_detected_gt_indices(gt_subset, md_results)
sahi_detected = get_detected_gt_indices(gt_subset, sahi_results)
all_detected = md_detected | sahi_detected
missed_mask = ~gt_subset.index.isin(all_detected)
missed = gt_subset[missed_mask]

print(f"Ground truth animals:        {len(gt_subset)}")
print(f"Detected by standard MD:     {len(md_detected)}")
print(f"Detected by SAHI:            {len(sahi_detected)}")
print(f"Missed by BOTH:              {len(missed)}")
print(f"→ {100 * len(missed) / len(gt_subset):.0f}% of animals are invisible to MegaDetector")
print(f"\nMissed by species:")
print(missed["species"].value_counts())

In [ ]:
# Visualize false negatives — zoomed crops of animals both MD and SAHI missed
# Since MD barely detects anything, most GT annotations are false negatives.
# Sample a diverse set across images for visual impact.

if len(missed) == 0:
    print("No false negatives — all animals were detected (unexpected!).")
else:
    # Sample up to 9 missed animals, spread across different images
    missed_shuffled = missed.sample(frac=1, random_state=42)
    missed_sample = missed_shuffled.drop_duplicates("filename").head(9)
    # If fewer than 9 images, fill with more from the same images
    if len(missed_sample) < 9:
        remaining = missed_shuffled[~missed_shuffled.index.isin(missed_sample.index)]
        missed_sample = pd.concat([missed_sample, remaining.head(9 - len(missed_sample))])

    n_show = min(9, len(missed_sample))
    cols = 3
    rows = (n_show + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes_flat = axes.flat if hasattr(axes, "flat") else [axes]

    pad = 80
    for idx, (ax, (_, row)) in enumerate(zip(axes_flat, missed_sample.head(n_show).iterrows())):
        img_arr = np.array(Image.open(TEST_IMG_DIR / row["filename"]))
        cx = (row["x1"] + row["x2"]) / 2
        cy = (row["y1"] + row["y2"]) / 2
        crop_x1 = max(0, int(cx - pad))
        crop_y1 = max(0, int(cy - pad))
        crop_x2 = min(img_arr.shape[1], int(cx + pad))
        crop_y2 = min(img_arr.shape[0], int(cy + pad))

        crop = img_arr[crop_y1:crop_y2, crop_x1:crop_x2]
        ax.imshow(crop)

        # Draw GT box (green) — the animal that was missed
        ax.add_patch(mpatches.Rectangle(
            (row["x1"] - crop_x1, row["y1"] - crop_y1), row["w"], row["h"],
            linewidth=2, edgecolor="#2ECC71", facecolor="none",
        ))
        species = row.get("species", "unknown")
        ax.set_title(f"{species} — MISSED\n{row['w']}x{row['h']} px", fontsize=9)
        ax.axis("off")

    # Hide unused subplots
    for j in range(n_show, rows * cols):
        axes_flat[j].axis("off") if j < len(list(axes_flat)) else None

    plt.suptitle("False negatives: animals that MegaDetector + SAHI both missed", fontsize=12)
    plt.tight_layout()

In [ ]:
def compute_iou(box_a, box_b):
    """Compute IoU between two boxes [x1, y1, x2, y2]."""
    x1 = max(box_a[0], box_b[0])
    y1 = max(box_a[1], box_b[1])
    x2 = min(box_a[2], box_b[2])
    y2 = min(box_a[3], box_b[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter

    return inter / union if union > 0 else 0.0


def match_detections(gt_df, predictions, iou_threshold=0.3):
    """
    Match predictions to ground truth per image.
    Returns DataFrame with TP, FP, FN counts per image.
    """
    results = []

    all_filenames = set(gt_df["filename"].unique())
    for pred in predictions:
        all_filenames.add(pred["filename"])

    for fname in sorted(all_filenames):
        img_gt = gt_df[gt_df["filename"] == fname]
        gt_boxes = img_gt[["x1", "y1", "x2", "y2"]].values.tolist()
        gt_matched = [False] * len(gt_boxes)

        pred_entry = next((r for r in predictions if r["filename"] == fname), None)
        pred_boxes = []
        if pred_entry:
            pred_boxes = [
                [d["x1"], d["y1"], d["x2"], d["y2"]]
                for d in pred_entry["detections"] if d["class"] == 0
            ]

        tp = 0
        fp = 0

        for pb in pred_boxes:
            best_iou = 0
            best_idx = -1
            for j, gb in enumerate(gt_boxes):
                if gt_matched[j]:
                    continue
                iou = compute_iou(pb, gb)
                if iou > best_iou:
                    best_iou = iou
                    best_idx = j

            if best_iou >= iou_threshold:
                tp += 1
                gt_matched[best_idx] = True
            else:
                fp += 1

        fn = sum(1 for m in gt_matched if not m)

        results.append({
            "filename": fname,
            "gt_count": len(gt_boxes),
            "pred_count": len(pred_boxes),
            "tp": tp, "fp": fp, "fn": fn,
        })

    return pd.DataFrame(results)

In [ ]:
# Compare metrics: Standard MD vs SAHI
# Compute standard MD metrics here so this cell is self-contained
md_metrics_df = match_detections(gt_subset, md_results, iou_threshold=0.3)
md_TP = md_metrics_df["tp"].sum()
md_FP = md_metrics_df["fp"].sum()
md_FN = md_metrics_df["fn"].sum()
md_precision = md_TP / (md_TP + md_FP) if (md_TP + md_FP) > 0 else 0
md_recall = md_TP / (md_TP + md_FN) if (md_TP + md_FN) > 0 else 0
md_f1 = 2 * md_precision * md_recall / (md_precision + md_recall) if (md_precision + md_recall) > 0 else 0
md_mae = (md_metrics_df["pred_count"] - md_metrics_df["gt_count"]).abs().mean()
md_total_pred = md_metrics_df["pred_count"].sum()

sahi_metrics_df = match_detections(gt_subset, sahi_results, iou_threshold=0.3)
sahi_TP = sahi_metrics_df["tp"].sum()
sahi_FP = sahi_metrics_df["fp"].sum()
sahi_FN = sahi_metrics_df["fn"].sum()
sahi_precision = sahi_TP / (sahi_TP + sahi_FP) if (sahi_TP + sahi_FP) > 0 else 0
sahi_recall = sahi_TP / (sahi_TP + sahi_FN) if (sahi_TP + sahi_FN) > 0 else 0
sahi_f1 = 2 * sahi_precision * sahi_recall / (sahi_precision + sahi_recall) if (sahi_precision + sahi_recall) > 0 else 0
sahi_mae = (sahi_metrics_df["pred_count"] - sahi_metrics_df["gt_count"]).abs().mean()
sahi_total_pred = sahi_metrics_df["pred_count"].sum()

comparison = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "MAE", "Total detections", "TP", "FP", "FN"],
    "Standard MD": [f"{md_precision:.3f}", f"{md_recall:.3f}", f"{md_f1:.3f}", f"{md_mae:.2f}",
                    str(md_total_pred), str(md_TP), str(md_FP), str(md_FN)],
    "MD + SAHI": [f"{sahi_precision:.3f}", f"{sahi_recall:.3f}", f"{sahi_f1:.3f}", f"{sahi_mae:.2f}",
                  str(sahi_total_pred), str(sahi_TP), str(sahi_FP), str(sahi_FN)],
})

print("=" * 60)
print(f"  {'Metric':<20s}  {'Standard MD':>12s}  {'MD + SAHI':>12s}")
print("=" * 60)
for _, row in comparison.iterrows():
    print(f"  {row['Metric']:<20s}  {row['Standard MD']:>12s}  {row['MD + SAHI']:>12s}")
print("=" * 60)
print(f"\n  Ground truth: {gt_total} animals")
print(f"  SAHI tile size: {SLICE_SIZE}x{SLICE_SIZE}, overlap: {OVERLAP_RATIO:.0%}")
print(f"  Time: standard {elapsed:.1f}s vs SAHI {sahi_elapsed:.1f}s")

## 5 — The domain shift verdict

MegaDetector — even with SAHI tiled inference — cannot detect animals
in nadir aerial imagery. The model was trained on side-view camera trap
photos where animals are large, centred, and at eye level. From above,
the same animals are tiny, top-down silhouettes against a uniform background.

**This is not a threshold problem or a resolution problem — it is a
fundamental domain mismatch.** The solution is to train a domain-specific
model on aerial data (see the training practical).

In [ ]:
# Confidence distribution of SAHI detections vs. ground truth overlap
# For each SAHI detection, compute its best IoU with any GT box

det_confs = []  # (conf, best_iou, is_tp)
for r in sahi_results:
    fname = r["filename"]
    img_gt = gt_subset[gt_subset["filename"] == fname]
    gt_boxes = img_gt[["x1", "y1", "x2", "y2"]].values.tolist()

    for d in r["detections"]:
        if d["class"] != 0:
            continue
        pred_box = [d["x1"], d["y1"], d["x2"], d["y2"]]
        best_iou = max((compute_iou(pred_box, gb) for gb in gt_boxes), default=0.0)
        det_confs.append({
            "conf": d["conf"],
            "best_iou": best_iou,
            "tp": best_iou >= 0.3,
        })

det_df = pd.DataFrame(det_confs)

if len(det_df) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Left: confidence histogram, colored by TP/FP
    ax = axes[0]
    tp_confs = det_df[det_df["tp"]]["conf"]
    fp_confs = det_df[~det_df["tp"]]["conf"]
    ax.hist([tp_confs, fp_confs], bins=20, stacked=True,
            color=["#2ECC71", "#E74C3C"], label=["True Positive", "False Positive"],
            edgecolor="white")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Number of detections")
    ax.set_title("SAHI detection confidence distribution")
    ax.legend()

    # Right: confidence vs IoU scatter
    ax = axes[1]
    colors = ["#2ECC71" if tp else "#E74C3C" for tp in det_df["tp"]]
    ax.scatter(det_df["conf"], det_df["best_iou"], c=colors, alpha=0.5, s=20)
    ax.axhline(0.3, color="gray", linestyle="--", alpha=0.5, label="IoU threshold")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Best IoU with GT")
    ax.set_title("Confidence vs. IoU overlap")
    ax.legend()

    plt.suptitle("Are confident detections correct? (SAHI results)", fontsize=12)
    plt.tight_layout()
else:
    print("No SAHI animal detections to analyze.")

In [ ]:
# Precision-Recall curve by sweeping confidence threshold
# For each threshold, compute P/R from SAHI detections vs GT

thresholds = np.arange(0.05, 0.95, 0.02)
pr_points = []

for t in thresholds:
    filtered = []
    for r in sahi_results:
        filtered.append({
            "filename": r["filename"],
            "detections": [d for d in r["detections"] if d["conf"] >= t],
        })

    mdf = match_detections(gt_subset, filtered, iou_threshold=0.3)
    tp = mdf["tp"].sum()
    fp = mdf["fp"].sum()
    fn = mdf["fn"].sum()
    p = tp / (tp + fp) if (tp + fp) > 0 else 1.0  # no predictions = perfect precision
    r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    pr_points.append({"threshold": t, "precision": p, "recall": r})

pr_df = pd.DataFrame(pr_points)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Precision-Recall curve
ax = axes[0]
ax.plot(pr_df["recall"], pr_df["precision"], "o-", color="steelblue", markersize=3)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall curve (SAHI + MegaDetector)")
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

# Annotate a few threshold values
for t_label in [0.1, 0.3, 0.5, 0.7]:
    row = pr_df.iloc[(pr_df["threshold"] - t_label).abs().argmin()]
    ax.annotate(f"t={t_label}", (row["recall"], row["precision"]),
                fontsize=7, color="red", textcoords="offset points", xytext=(5, 5))

# Right: P, R, F1 vs threshold
ax = axes[1]
pr_df["f1"] = 2 * pr_df["precision"] * pr_df["recall"] / (pr_df["precision"] + pr_df["recall"]).replace(0, 1e-8)
ax.plot(pr_df["threshold"], pr_df["precision"], label="Precision", color="#E74C3C")
ax.plot(pr_df["threshold"], pr_df["recall"], label="Recall", color="#3498DB")
ax.plot(pr_df["threshold"], pr_df["f1"], label="F1", color="#2ECC71", linewidth=2)
ax.set_xlabel("Confidence threshold")
ax.set_ylabel("Score")
ax.set_title("P / R / F1 vs. confidence threshold")
ax.legend()
ax.set_ylim(-0.05, 1.05)
ax.grid(alpha=0.3)

# Find best F1
best_row = pr_df.loc[pr_df["f1"].idxmax()]
ax.axvline(best_row["threshold"], color="gray", linestyle="--", alpha=0.5)
ax.annotate(f"best F1={best_row['f1']:.3f}\nt={best_row['threshold']:.2f}",
            (best_row["threshold"], best_row["f1"]),
            fontsize=8, textcoords="offset points", xytext=(10, -15))

plt.suptitle("MegaDetector + SAHI on aerial data: no threshold saves the model", fontsize=12)
plt.tight_layout()

## Exercises

1. **Resolution experiment** — Re-run MegaDetector with `imgsz=1280`:
   ```python
   results = md_model.predict(str(img_path), conf=0.1, imgsz=1280, verbose=False)
   ```
   Does doubling the input resolution help? How much? What is the speed trade-off?

2. **SAHI tile size** — We used 640x640 tiles. Try 512x512 and 1024x1024.
   Does smaller tile size improve recall (animals are larger per tile) or
   just increase false positives? What is the optimal tile size for this data?

3. **Try a different detector** — Load MDv6 (RT-DETR based) instead of v1000 larch:
   ```python
   from ultralytics import RTDETR
   mdv6 = RTDETR("path/to/MDV6-rtdetr-c.pt")
   ```
   Or try the sorrel variant (YOLOv11-S, 9M params):
   ```python
   mdv6_sorrel = YOLO("path/to/md_v1000.0.0-sorrel.pt")
   ```
   Does a different architecture change anything, or is the domain shift
   too fundamental for any camera-trap-trained model?

4. **False positive analysis** — Look at the zoomed SAHI and high-confidence
   detection crops from section 4b. What is MegaDetector detecting?
   Trees? Shadows? Rocks? What does this tell you about what
   the model learned from camera trap data?

5. **Compare with Eikelboom's RetinaNet** — The paper reports 90–95% recall
   with a RetinaNet trained on this aerial data. Look at the PR curve from
   section 5 — what is the best recall MegaDetector can achieve at any threshold?
   What does the gap tell you about when retraining is necessary vs. when
   tuning is enough?

## Reflection

- MegaDetector is the most widely used model in wildlife AI, but it fails on
  aerial imagery. What does this tell us about the limits of transfer learning?
  When can we trust a model to generalise to new data?

- The Eikelboom paper trained a RetinaNet specifically on this aerial data and
  achieved 90–95% recall. What changed between MegaDetector and a domain-specific
  model? (Training data, viewpoint, object size, background.)

- The animals here are 30–60 px in a 14 MP image. At YOLO's default 640 px input,
  each animal is 4–8 pixels. Is this a model problem or a resolution problem?
  How does SAHI address this?

- If you were starting a new aerial wildlife survey, would you:
  (a) fine-tune MegaDetector on your aerial data,
  (b) train a new detector from scratch (like Eikelboom's RetinaNet),
  (c) use a point-based detector like HerdNet?
  What factors would influence your decision?

In [ ]:
##TODO Now add Herdnet with the same Dataset and compare results
